# Generate perturbations

Load the allowed training subset and select one robust and one non-robust problem for `Qwen/Qwen3.5-4B`. Null labels are excluded from selection. Generate variants of the non-robust problem using the Codex perturbation helper.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").is_file()
)
dataset_path = REPO_ROOT / "data_artifacts" / "train-main-v2" / "train_allowed.parquet"
train_allowed = pd.read_parquet(dataset_path)

qwen_cases = train_allowed.loc[
    train_allowed["model_id"].eq("Qwen/Qwen3.5-4B")
].copy()
qwen_cases["is_robust"] = qwen_cases["is_robust"].astype("boolean")
print(f"Qwen cases: {len(qwen_cases)}")
display(qwen_cases["is_robust"].value_counts(dropna=False))

Qwen cases: 7


is_robust
True     4
False    2
<NA>     1
Name: count, dtype: Int64

In [2]:
# Select the first case in file order for each label.
robust_case = qwen_cases.loc[
    qwen_cases["is_robust"].eq(True).fillna(False)
].iloc[0]
non_robust_case = qwen_cases.loc[
    qwen_cases["is_robust"].eq(False).fillna(False)
].iloc[0]

robust_problem = robust_case["problem"]
non_robust_problem = non_robust_case["problem"]

In [3]:
display(Markdown("## Robust problem\n\n" + robust_problem))
display(Markdown("## Non-robust problem\n\n" + non_robust_problem))

## Robust problem

The Fibonacci numbers are defined as follows: $F_0 = 0$, $F_1 = 1$, and $F_{n+1} = F_n + F_{n-1}$ for $n \geq 1$.
There are $N$ positive integers $n$ strictly less than $10^{101}$ such that $n^2 + (n+1)^2$ is a multiple of 5
but $F_{n-1}^2 + F_n^2$ is not. How many prime factors does $N$ have, counted with multiplicity?

## Non-robust problem

Let $S$ denote the value of the infinite sum
\[
\frac{1}{9} + \frac{1}{99} + \frac{1}{999} + \frac{1}{9999}  + \cdots
\]
Find the remainder when the greatest integer less than or equal to $10^{100} S$ is divided by $1000.$

## Perturb the non-robust problem

Create one independent variant for each perturbation type. Running the next cell makes five Codex calls. The variants request answer preservation but have not been mathematically verified.

In [4]:
from aimo_interp import PerturbationType, perturb_question

perturbation_types = list(PerturbationType)
non_robust_variants = perturb_question(non_robust_problem, perturbation_types)

In [5]:
for variant in non_robust_variants:
    display(Markdown(f"## {variant.perturbation_type.value.title()}\n\n{variant.reply}"))

## Rephrase

Define $S$ by the infinite series
\[
S=\frac{1}{9}+\frac{1}{99}+\frac{1}{999}+\frac{1}{9999}+\cdots.
\]
What remainder is obtained upon dividing the greatest integer less than or equal to $10^{100}S$ by $1000$?

## Rename

Let $T$ denote the value of the infinite sum
\[
\frac{1}{9} + \frac{1}{99} + \frac{1}{999} + \frac{1}{9999} + \cdots
\]
Find the remainder when the greatest integer less than or equal to $10^{100} T$ is divided by $1000.$

## Domain

In a mathematical accounting model, successive entries have values
\[
\frac{1}{9},\quad \frac{1}{99},\quad \frac{1}{999},\quad \frac{1}{9999},\quad \ldots
\]
Let $S$ denote the sum of all infinitely many entries. Find the remainder when the greatest integer less than or equal to $10^{100}S$ is divided by $1000.$

## Distract

A mathematics club includes the following problem in its weekly newsletter. Let $S$ denote the value of the infinite sum
\[
\frac{1}{9} + \frac{1}{99} + \frac{1}{999} + \frac{1}{9999} + \cdots
\]
Find the remainder when the greatest integer less than or equal to $10^{100} S$ is divided by $1000.$

## Typos

Let $S$ denote the value of the infinite sum
\[
\frac{1}{9} + \frac{1}{99}  + \frac{1}{999} + \frac{1}{9999}  + \cdots
\]
Find the remainder when the greatest integer less than or equal to $10^{100} S$ is divded by $1000.$

## Perturb all training problems

Read the full `train.parquet` (all models), keeping each unique problem once.
Generate all five perturbation types into `train_perturbed_df` and save
`data_artifacts/prompt_perturbations/train_perturbed.parquet` with the columns
`original_problem`, `perturbation_type`, and `perturbed_problem`.

This makes up to 180 Codex calls, with four running concurrently. Each completed
variant is saved atomically; rerunning skips saved problem/type pairs. Generated
variants have not been mathematically verified.

In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd
from aimo_interp import PerturbationType, perturb_question

REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").is_file()
)
train_path = REPO_ROOT / "data/train-main-v2/train.parquet"
output_path = REPO_ROOT / "data_artifacts/prompt_perturbations/train_perturbed.parquet"
output_path.parent.mkdir(parents=True, exist_ok=True)

original_problems = pd.read_parquet(train_path)["problem"].dropna().drop_duplicates().tolist()
columns = ["original_problem", "perturbation_type", "perturbed_problem"]
train_perturbed_df = (
    pd.read_parquet(output_path)[columns]
    if output_path.exists() else pd.DataFrame(columns=columns)
)
rows = train_perturbed_df.to_dict("records")
completed = set(zip(train_perturbed_df["original_problem"], train_perturbed_df["perturbation_type"]))
jobs = [
    (problem, kind)
    for problem in original_problems
    for kind in PerturbationType
    if (problem, kind.value) not in completed
]
print(f"{len(original_problems)} unique problems; {len(jobs)} variants remaining.", flush=True)
failures = []
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {
        executor.submit(perturb_question, problem, [kind]): (problem, kind)
        for problem, kind in jobs
    }
    for future in as_completed(futures):
        problem, kind = futures[future]
        try:
            variant = future.result()[0]
        except Exception as exc:
            failures.append((problem, kind.value, str(exc)))
            print(f"Failed {kind.value}: {exc}", flush=True)
            continue
        rows.append({
            "original_problem": problem,
            "perturbation_type": variant.perturbation_type.value,
            "perturbed_problem": variant.reply,
        })
        train_perturbed_df = pd.DataFrame(rows, columns=columns)
        checkpoint = output_path.with_suffix(".tmp.parquet")
        train_perturbed_df.to_parquet(checkpoint, index=False)
        checkpoint.replace(output_path)
        print(f"Saved {len(rows)}/{len(original_problems) * len(PerturbationType)} variants.", flush=True)

print(f"Saved {len(train_perturbed_df)} rows to {output_path}")
if failures:
    raise RuntimeError(f"{len(failures)} variants failed. Rerun this cell to retry missing variants.")
display(train_perturbed_df.head())

36 unique problems; 180 variants remaining.


Saved 1/180 variants.


Saved 2/180 variants.


Saved 3/180 variants.


Saved 4/180 variants.


...

Saved 178/180 variants.


Saved 179/180 variants.


Saved 180/180 variants.


Saved 180 rows to /home/suhas-shankar/Projects/AI_Research/AIMO_Interp/aimo-interp-investigate-black-box/data_artifacts/prompt_perturbations/train_perturbed.parquet


,original_problem,perturbation_type,perturbed_problem
0,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...
1,Alice and Bob are each holding some integer nu...,distract,Alice and Bob are each holding some integer nu...
2,Alice and Bob are each holding some integer nu...,domain,"Two game characters, Mira and Theo, each have ..."
3,Alice and Bob are each holding some integer nu...,rephrase,Alice and Bob each have an integer number of s...
4,Alice and Bob are each holding some integer nu...,typos,Alice and Bob are each holding some integer nu...


In [2]:
train_perturbed_df = pd.read_parquet(REPO_ROOT / "data_artifacts/prompt_perturbations/train_perturbed.parquet")

In [8]:
train_perturbed_df

,original_problem,perturbation_type,perturbed_problem
0,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...
1,Alice and Bob are each holding some integer nu...,distract,Alice and Bob are each holding some integer nu...
2,Alice and Bob are each holding some integer nu...,domain,"Two game characters, Mira and Theo, each have ..."
3,Alice and Bob are each holding some integer nu...,rephrase,Alice and Bob each have an integer number of s...
4,Alice and Bob are each holding some integer nu...,typos,Alice and Bob are each holding some integer nu...
...,...,...,...
175,"On a blackboard, Ken starts off by writing a p...",typos,"On a blackboard, Ken starts off by writing a p..."
176,Let $ABC$ be an acute-angled triangle with int...,rename,Let $PQR$ be an acute-angled triangle with int...
177,Let $ABC$ be an acute-angled triangle with int...,domain,A geometric design for a mosaic uses an acute-...
178,Let $ABC$ be an acute-angled triangle with int...,typos,Let $ABC$ be an acute-angled triangle with int...


In [3]:
from aimo_interp.preprocessing import generate_problem_id


train_perturbed_df["problem_id"] = generate_problem_id(
    train_perturbed_df["original_problem"]
)

In [11]:
train_perturbed_df

,original_problem,perturbation_type,perturbed_problem,problem_id
0,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...
1,Alice and Bob are each holding some integer nu...,distract,Alice and Bob are each holding some integer nu...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...
2,Alice and Bob are each holding some integer nu...,domain,"Two game characters, Mira and Theo, each have ...",2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...
3,Alice and Bob are each holding some integer nu...,rephrase,Alice and Bob each have an integer number of s...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...
4,Alice and Bob are each holding some integer nu...,typos,Alice and Bob are each holding some integer nu...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...
...,...,...,...,...
175,"On a blackboard, Ken starts off by writing a p...",typos,"On a blackboard, Ken starts off by writing a p...",57b6ba21f3f5c26b956319d742488e54b89e96d8f57c7c...
176,Let $ABC$ be an acute-angled triangle with int...,rename,Let $PQR$ be an acute-angled triangle with int...,514ab5381d43028c56de1d2462da06354913886d6e2e6a...
177,Let $ABC$ be an acute-angled triangle with int...,domain,A geometric design for a mosaic uses an acute-...,514ab5381d43028c56de1d2462da06354913886d6e2e6a...
178,Let $ABC$ be an acute-angled triangle with int...,typos,Let $ABC$ be an acute-angled triangle with int...,514ab5381d43028c56de1d2462da06354913886d6e2e6a...


## Join original robustness labels

Match the original problem hashes to the full training set. Each variant is
repeated for every model/effort configuration that has that original problem.
`is_robust` and `max_drop` describe the **original problem** for that configuration,
not the generated variant. Keep null labels as unknown.

Save the joined dataframe separately as `train_perturbed_labeled.parquet`.

In [13]:
train_labels_df = pd.read_parquet(REPO_ROOT / "data/train-main-v2/train.parquet")
train_labels_df["problem_id"] = generate_problem_id(train_labels_df["problem"])
train_labels_df["is_robust"] = train_labels_df["is_robust"].astype("boolean")
case_keys = ["problem_id", "model_id", "reasoning_effort"]
assert not train_labels_df.duplicated(case_keys).any(), "Duplicate model/problem/effort cases."

train_perturbed_labeled_df = train_perturbed_df.merge(
    train_labels_df[["problem_id", "model_id", "reasoning_effort", "is_robust", "max_drop"]],
    on="problem_id",
    how="left",
    validate="many_to_many",  # Multiple variants and multiple model labels per original.
    indicator=True,
)
assert train_perturbed_labeled_df["_merge"].eq("both").all(), "Some original problems have no training row."
train_perturbed_labeled_df = train_perturbed_labeled_df.drop(columns="_merge")

labeled_output_path = REPO_ROOT / "data_artifacts/prompt_perturbations/train_perturbed_labeled.parquet"
train_perturbed_labeled_df.to_parquet(labeled_output_path, index=False)
print(f"Saved {len(train_perturbed_labeled_df)} rows to {labeled_output_path}")
print(train_perturbed_labeled_df["is_robust"].value_counts(dropna=False))
display(train_perturbed_labeled_df.head())

Saved 410 rows to /home/suhas-shankar/Projects/AI_Research/AIMO_Interp/aimo-interp-investigate-black-box/data_artifacts/prompt_perturbations/train_perturbed_labeled.parquet
is_robust
True     260
False     95
<NA>      55
Name: count, dtype: Int64


,original_problem,perturbation_type,perturbed_problem,problem_id,model_id,reasoning_effort,is_robust,max_drop
0,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...,Qwen/Qwen3.5-4B,default,True,-0.05
1,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...,deepseek-ai/DeepSeek-R1-0528-Qwen3-8B,default,True,0.08
2,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...,openai/gpt-oss-120b,high,True,0.01
3,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...,openai/gpt-oss-120b,low,True,0.07
4,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...,openai/gpt-oss-120b,medium,True,0.01


In [14]:
train_perturbed_labeled_df

,original_problem,perturbation_type,perturbed_problem,problem_id,model_id,reasoning_effort,is_robust,max_drop
0,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...,Qwen/Qwen3.5-4B,default,True,-0.05
1,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...,deepseek-ai/DeepSeek-R1-0528-Qwen3-8B,default,True,0.08
2,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...,openai/gpt-oss-120b,high,True,0.01
3,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...,openai/gpt-oss-120b,low,True,0.07
4,Alice and Bob are each holding some integer nu...,rename,Clara and Daniel are each holding some integer...,2ca839cfbfc60e29b4b761e290c8a294243b03efe16b78...,openai/gpt-oss-120b,medium,True,0.01
...,...,...,...,...,...,...,...,...
405,"On a blackboard, Ken starts off by writing a p...",typos,"On a blackboard, Ken starts off by writing a p...",57b6ba21f3f5c26b956319d742488e54b89e96d8f57c7c...,openai/gpt-oss-120b,medium,False,0.30
406,Let $ABC$ be an acute-angled triangle with int...,rename,Let $PQR$ be an acute-angled triangle with int...,514ab5381d43028c56de1d2462da06354913886d6e2e6a...,openai/gpt-oss-120b,medium,False,0.44
407,Let $ABC$ be an acute-angled triangle with int...,domain,A geometric design for a mosaic uses an acute-...,514ab5381d43028c56de1d2462da06354913886d6e2e6a...,openai/gpt-oss-120b,medium,False,0.44
408,Let $ABC$ be an acute-angled triangle with int...,typos,Let $ABC$ be an acute-angled triangle with int...,514ab5381d43028c56de1d2462da06354913886d6e2e6a...,openai/gpt-oss-120b,medium,False,0.44
